## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 105 samples and 15151 columns.
First few rows of the data:


,overall survival period from diagnosis date (year),overall survival event,overall survival period from resection date (year),overall survival event.1,Sample,gender,age at diagnosis (years),mutation category,subtype,primary tumor location,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
0,4.1863,0.0,4.0822,0.0,BAIX,M,48.0,BRAF,SunExposed,right occipital lobe (scalp),...,4.238683,5.644566,6.233697,1.819664,2.430716,5.270749,1.157515,5.129748,6.262725,6.327488
1,3.4575,0.0,2.1808,0.0,BAY,M,63.0,WT,SunExposed,right lateral arm,...,3.369043,5.044671,5.661581,2.281543,2.968554,4.835667,1.755134,4.864588,4.854396,5.872085
2,5.1616,0.0,1.3425,0.0,BILI,M,39.0,WT,SunExposed,left ear (tragus),...,3.831359,4.721497,4.685914,2.305708,2.726233,4.244033,0.718545,4.291452,4.525052,5.248131
3,9.3562,0.0,3.3534,0.0,BROT,M,71.0,WT,SunExposed,right shoulder,...,4.318865,5.368309,5.091451,1.646034,2.601106,3.944951,0.920962,4.061010,7.017117,5.386654
4,8.6767,1.0,0.2795,1.0,CHER,M,55.0,RAS,SunExposed,right arm,...,3.278025,4.431412,5.114594,2.005626,2.485849,4.602899,0.851397,4.473105,7.566240,5.258804


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'Sample'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
